# CC4: tracing the FT chart

The brief allows the data for a replica to be traced by hand. This notebook traces it from the pixels instead: it reads the published image, finds every bar, and measures where each coloured segment starts and ends.

**Chart:** Financial Times, “A shrinking share of capital taxes” (Martin Sandbu, Free Lunch, 5 November 2026). Data behind it: Luxembourg Income Study working paper 895 (Péligry and Ragot).

Outputs: `cc4_ft_tax_shares.csv` (the replica's data) and `cc4_capital_tax_change.csv` (the improved chart's data).

In [ ]:
import io, urllib.request
import numpy as np
import pandas as pd
from PIL import Image

BASE = "https://raw.githubusercontent.com/jakobbilan/jakobbilan.github.io/main/img/"

def load(name):
    with urllib.request.urlopen(BASE + name) as response:
        return np.asarray(Image.open(io.BytesIO(response.read())).convert("RGB")).astype(int)

shares_img = load("cc4_original.png")        # the chart on the page: % of all tax revenue
levels_img = load("cc4_companion_gdp.png")   # its companion in the same article: capital taxes, % of GDP
print(shares_img.shape, levels_img.shape)

## 1. The chart's colours and layout

Colours are sampled from the legend swatches. The axis ticks fix the scale: 0 sits at x = 127 and 100 at x = 1,002 (8.75 pixels per percentage point). The 16 bars are 17 pixels tall, 20 pixels apart, starting at y = 140.

In [ ]:
PAPER = np.array([255, 241, 230])
TAXES = {
    "consumption": ("Consumption taxes", (34, 84, 133)),
    "labour_households": ("Labour taxes paid by households", (149, 210, 228)),
    "labour_corporations": ("Labour taxes paid by corporations and self-employed", (39, 140, 199)),
    "capital_income_households": ("Capital income taxes paid by households", (181, 53, 92)),
    "capital_income_corporations": ("Capital income taxes paid by corporations and self-employed", (249, 120, 159)),
    "capital_stocks": ("Taxes on capital stocks", (170, 205, 113)),
}
LABELS = ["US 2006", "US 2019", "Japan 2006", "Japan 2019", "Germany 2006", "Germany 2019", "UK 2006", "UK 2019",
          "France 2006", "France 2019", "Italy 2006", "Italy 2019", "Canada 2006", "Canada 2019",
          "South Korea 2006", "South Korea 2019"]
X0, X100 = 127.0, 1002.0
BARS = [(140 + 20 * i, 156 + 20 * i) for i in range(16)]

## 2. Segment boundaries

Along each scan line inside a bar, every pixel is matched to the nearest legend colour; runs shorter than 3 pixels are the thin outlines between segments. A boundary is the midpoint between two runs. Each bar is scanned on 9 lines and the boundaries averaged.

In [ ]:
def scan(img, y, names, colours, x_from=122):
    row = img[y]
    xs = np.arange(x_from, img.shape[1])
    classes = []
    for x in xs:
        p = row[x]
        if np.abs(p - PAPER).sum() < 25:
            classes.append(None)
            continue
        d = np.abs(colours - p).sum(axis=1)
        classes.append(names[d.argmin()] if d.min() < 60 else "edge")
    runs, i = [], 0
    while i < len(classes):
        j = i
        while j < len(classes) and classes[j] == classes[i]:
            j += 1
        if classes[i] in names and j - i >= 3:
            runs.append([classes[i], xs[i], xs[j - 1]])
        i = j
    merged = []
    for r in runs:
        if merged and merged[-1][0] == r[0]:
            merged[-1][2] = r[2]
        else:
            merged.append(r)
    inner = [(merged[k][2] + merged[k + 1][1]) / 2 for k in range(len(merged) - 1)]
    return [m[0] for m in merged], inner, merged[-1][2] + 1.0

names = list(TAXES)
colours = np.array([TAXES[n][1] for n in names])
rows = []
for (y0, y1), label in zip(BARS, LABELS):
    lines = [scan(shares_img, y, names, colours) for y in range(y0 + 4, y1 - 3)]
    order = lines[0][0]
    assert all(l[0] == order for l in lines), label
    inner = np.mean([l[1] for l in lines], axis=0)
    edges = [0.0] + [(x - X0) / (X100 - X0) * 100 for x in inner] + [100.0]
    shares = [round(edges[k + 1] - edges[k], 1) for k in range(len(order))]
    k = int(np.argmax(shares))
    shares[k] = round(shares[k] + round(100 - sum(shares), 1), 1)    # rounding residue: bars sum to 100.0
    country, year = label.rsplit(" ", 1)
    for i, (tax, share) in enumerate(zip(order, shares)):
        rows.append({"country": country, "year": int(year), "label": label, "tax": tax,
                     "tax_label": TAXES[tax][0], "order": i + 1, "share": share})
tidy = pd.DataFrame(rows)
assert (tidy.groupby("label")["share"].sum().round(1) == 100.0).all()
tidy.to_csv("cc4_ft_tax_shares.csv", index=False)
tidy.pivot_table(index=["country", "year"], columns="tax", values="share", sort=False)

## 3. The companion chart, % of GDP

Same method; its axis runs from 0 at x = 127 to 10 at x = 1,007 (88 pixels per point). Dividing these levels by the shares above gives each country's total tax take, a check on both tracings: the results land on OECD-like totals (France about 44% of GDP, the US about 25–27%, South Korea 23% rising to 27%).

In [ ]:
LEVEL_COLOURS = {"capital_income_households": (30, 83, 137), "capital_income_corporations": (151, 208, 233),
                 "capital_stocks": (35, 140, 198)}
lnames = list(LEVEL_COLOURS)
lcolours = np.array([LEVEL_COLOURS[n] for n in lnames])
LEVEL_BARS = [(126, 167), (181, 222), (236, 277), (291, 332), (346, 387), (401, 441), (456, 496), (511, 551),
              (566, 606), (620, 661), (676, 716), (730, 771), (786, 826), (840, 881), (896, 936), (950, 991)]
levels = {}
for (y0, y1), label in zip(LEVEL_BARS, LABELS):
    lines = [scan(levels_img, y, lnames, lcolours) for y in range(y0 + 6, y1 - 5)]
    inner = np.mean([l[1] for l in lines], axis=0)
    right = np.mean([l[2] for l in lines])
    edges = [127.0] + list(inner) + [right]
    levels[label] = round((edges[-1] - edges[0]) / 88.0, 1)        # capital taxes, % of GDP
levels

## 4. One row per country for the improved chart

Capital taxes are the three capital categories added up. The change between 2006 and 2019 is split by tax; the three parts add up to the total.

In [ ]:
wide = tidy.pivot_table(index=["country", "year"], columns="tax", values="share", sort=False)
CAPITAL = ["capital_income_households", "capital_income_corporations", "capital_stocks"]
out = []
for country in ["US", "Japan", "Germany", "UK", "France", "Italy", "Canada", "South Korea"]:
    a, b = wide.loc[(country, 2006)], wide.loc[(country, 2019)]
    cap_a, cap_b = round(a[CAPITAL].sum(), 1), round(b[CAPITAL].sum(), 1)
    g_a, g_b = levels[f"{country} 2006"], levels[f"{country} 2019"]
    out.append({
        "country": country,
        "capital_share_2006": cap_a, "capital_share_2019": cap_b, "change": round(cap_b - cap_a, 1),
        "change_households_capital_income": round(b.capital_income_households - a.capital_income_households, 1),
        "change_company_capital_income": round(b.capital_income_corporations - a.capital_income_corporations, 1),
        "change_capital_stocks": round(b.capital_stocks - a.capital_stocks, 1),
        "labour_share_2006": round(a.labour_households + a.labour_corporations, 1),
        "labour_share_2019": round(b.labour_households + b.labour_corporations, 1),
        "consumption_share_2006": a.consumption, "consumption_share_2019": b.consumption,
        "capital_tax_gdp_2006": g_a, "capital_tax_gdp_2019": g_b,
        "total_tax_gdp_2006": round(100 * g_a / cap_a, 1), "total_tax_gdp_2019": round(100 * g_b / cap_b, 1),
    })
change = pd.DataFrame(out)
parts = change[["change_households_capital_income", "change_company_capital_income", "change_capital_stocks"]].sum(axis=1)
assert (abs(parts - change["change"]) < 0.051).all()
change.to_csv("cc4_capital_tax_change.csv", index=False)
change.sort_values("change")

## 5. Download

Put both files in the site's `data` folder.

In [ ]:
try:
    from google.colab import files
    files.download("cc4_ft_tax_shares.csv")
    files.download("cc4_capital_tax_change.csv")
except ImportError:
    print("Not in Colab: the files are in the working folder.")